In [0]:
#| default_exp foundation

## Foundation

Small shared mechanics: CLI behavior, failure telemetry, cell parsing, validation, and chapter navigation.

In [0]:
#| export
import ast
import hashlib
import json
import os
import re
import sys
import time
from contextlib import contextmanager
from functools import wraps
from pathlib import Path

from fastcore.nbio import mk_cell
from fastcore.script import _in_call_parse

In [ ]:
#| export
def _cli_return(value=None):
    return None if _in_call_parse.get() else value

In [ ]:
#| export
def _cli_error(msg):
    if _in_call_parse.get():
        print(msg, file=sys.stderr)
        raise SystemExit(1)
    raise ValueError(msg)

In [ ]:
#| export
def _failure_map_path():
    default = Path.home() / ".nbskill" / "nbskill-errors.json"
    return Path(os.environ.get("NBSKILL_FAILURE_MAP", default)).expanduser()

In [ ]:
#| export
def _empty_failure_map():
    return {"version": 1, "events": [], "counts": {}, "last_call": None}

In [ ]:
#| export
def _load_failure_map(path):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        data = _empty_failure_map()
    data.setdefault("version", 1)
    data.setdefault("events", [])
    data.setdefault("counts", {})
    data.setdefault("last_call", None)
    return data

In [ ]:
#| export
def _bump_count(data, kind, tool):
    counts = data.setdefault("counts", {})
    group = counts.setdefault(kind, {})
    group[tool] = group.get(tool, 0) + 1

In [ ]:
#| export
def _write_failure_map(path, data):
    data["events"] = data.get("events", [])[-200:]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

In [ ]:
#| export
def _record_tool_start(tool):
    path = _failure_map_path()
    now = time.time()
    event = {"tool": tool, "ts": now}
    try:
        data = _load_failure_map(path)
        last = data.get("last_call")
        if last:
            delta = now - float(last.get("ts", now))
            reasons = []
            if last.get("tool") == tool: reasons.append("same_tool")
            if delta <= 1.0: reasons.append("within_1s")
            if reasons:
                _bump_count(data, "friction", tool)
                data["events"].append({
                    "kind": "friction",
                    "tool": tool,
                    "previous_tool": last.get("tool"),
                    "seconds_since_previous": round(delta, 3),
                    "reasons": reasons,
                    "ts": now,
                })
        data["last_call"] = event
        _write_failure_map(path, data)
    except OSError:
        pass
    return event

In [ ]:
#| export
def _record_tool_failure(event, exc):
    path = _failure_map_path()
    try:
        data = _load_failure_map(path)
        tool = event["tool"]
        _bump_count(data, "failures", tool)
        data["events"].append({
            "kind": "failure",
            "tool": tool,
            "error_type": type(exc).__name__,
            "error": str(exc),
            "ts": time.time(),
        })
        _write_failure_map(path, data)
    except OSError:
        pass

In [ ]:
#| export
@contextmanager
def _track_tool(tool):
    event = _record_tool_start(tool)
    try:
        yield
    except BaseException as exc:
        _record_tool_failure(event, exc)
        raise

In [ ]:
#| export
def _tracked_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with _track_tool(func.__name__):
            return func(*args, **kwargs)
    return wrapper

In [ ]:
#| export
def _parse_literal(value):
    if value is None: return None
    if isinstance(value, str):
        value = value.strip()
        if value.lower() in {"", "none", "null"}: return None
        try: return ast.literal_eval(value)
        except (SyntaxError, ValueError): return value
    return value

In [ ]:
#| export
def _none_if_string(value):
    return None if isinstance(value, str) and value.strip().lower() in {"", "none", "null"} else value

In [ ]:
#| export
def _parse_slice(value):
    if not isinstance(value, str) or ":" not in value: return None
    parts = value.split(":")
    if len(parts) not in (2, 3): return None
    vals = [int(p) if p else None for p in parts]
    return slice(*vals)

In [ ]:
#| export
def _as_index(value, length):
    idx = int(value)
    if idx < 0: idx += length
    if idx < 0 or idx >= length: raise IndexError(value)
    return idx

In [ ]:
#| export
def _parse_read_selector(value):
    value = _parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)): return [int(o) for o in value]
    return int(value)

In [ ]:
#| export
def _parse_write_target(value):
    value = _parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)):
        if len(value) != 2: raise ValueError("write ranges must have start and stop")
        return slice(value[0], value[1])
    return int(value)

In [ ]:
#| export
def _select_cells(cells, selector):
    items = list(enumerate(cells))
    if selector is None: return items
    selector = _parse_read_selector(selector)
    if isinstance(selector, slice): return items[selector]
    if isinstance(selector, list):
        return [(idx, cells[idx]) for idx in (_as_index(o, len(cells)) for o in selector)]
    idx = _as_index(selector, len(cells))
    return [(idx, cells[idx])]

In [ ]:
#| export
def _delete_cells(cells, selector):
    if selector is None: return
    target = _parse_write_target(selector)
    if isinstance(target, slice):
        del cells[target]
        return
    idx = _as_index(target, len(cells))
    del cells[idx]

In [ ]:
#| export
def _split_blocks(text):
    text = "" if text is None else str(text)
    if not text: return []
    return [o.strip("\n") for o in re.split(r"(?m)^\s*---\s*$", text) if o.strip()]

In [ ]:
#| export
def _coerce_cell(cell, default_type="code"):
    if isinstance(cell, dict): return cell
    if isinstance(cell, (tuple, list)) and len(cell) == 2:
        cell_type, source = cell
        return mk_cell(str(source), cell_type=str(cell_type))
    return mk_cell(str(cell), cell_type=default_type)

In [ ]:
#| export
def _is_definition_node(node):
    return isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))

In [ ]:
#| export
def _node_start_line(node):
    return min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1

In [ ]:
#| export
def _is_export_directive(line):
    return re.match(r"^\s*#\|\s*(export|exports|exporti)(\s|$)", line) is not None

In [ ]:
#| export
def _is_export_gap(lines):
    return bool(lines) and all((not line.strip()) or _is_export_directive(line) for line in lines)

In [ ]:
#| export
def _emit_code_chunk(chunks, lines, export_prefix=None):
    if export_prefix: lines = [*export_prefix, *lines]
    text = "\n".join(lines).strip("\n")
    if text: chunks.append(text)

In [ ]:
#| export
def _split_code_cell_sources(source):
    source = source.strip("\n")
    if not source: return []
    try: tree = ast.parse(source)
    except SyntaxError: return [source]
    if sum(1 for node in tree.body if _is_definition_node(node)) <= 1: return [source]

    lines = source.splitlines()
    first_start = _node_start_line(tree.body[0]) if tree.body else 0
    leading = lines[:first_start]
    shared_export = [line for line in leading if _is_export_directive(line)] if _is_export_gap(leading) else []
    chunks = []
    if shared_export:
        cursor = first_start
    else:
        _emit_code_chunk(chunks, leading)
        cursor = first_start

    for node in tree.body:
        start = _node_start_line(node)
        end = node.end_lineno
        gap = lines[cursor:start]
        if shared_export and _is_export_gap(gap): gap = []
        _emit_code_chunk(chunks, [*gap, *lines[start:end]], shared_export or None)
        cursor = end
    _emit_code_chunk(chunks, lines[cursor:])
    return chunks or [source]

In [ ]:
#| export
def _split_code_cell(cell):
    cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
    if cell_type != "code": return [cell]
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): source = "".join(source)
    sources = _split_code_cell_sources(str(source))
    if len(sources) <= 1: return [cell]
    return [mk_cell(source, cell_type="code") for source in sources]

In [ ]:
#| export
def _split_symbol_cells(cells):
    split = []
    for cell in cells: split.extend(_split_code_cell(cell))
    return split

In [ ]:
#| export
def _cell_source(cell):
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): return "".join(source)
    return str(source)

In [ ]:
#| export
def _cell_hash(cell_or_source, n=12):
    source = _cell_source(cell_or_source) if not isinstance(cell_or_source, str) else cell_or_source
    digest = hashlib.sha256(source.encode("utf-8")).hexdigest()
    return digest if n is None else digest[:n]

In [ ]:
#| export
def _parse_one_cell(text, default_type="code"):
    blocks = _split_blocks(text)
    if len(blocks) != 1: _cli_error("update_cell expects exactly one replacement cell")
    return _coerce_cell(blocks[0], default_type)

In [ ]:
#| export
def _cell_matches_hash(cell, source_hash):
    if source_hash is None: return True
    return _cell_hash(cell, n=None).startswith(str(source_hash).lower())

In [ ]:
#| export
def _find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == cell_id]
    if len(matches) == 1: return matches[0]
    if not matches: _cli_error(f"No cell has id {cell_id!r}")
    _cli_error(f"Multiple cells have id {cell_id!r}")

In [ ]:
#| export
def _find_cell_by_text(cells, old_str):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if old_str in _cell_source(cell)]
    if len(matches) == 1: return matches[0]
    if not matches: _cli_error("old_str did not match any cell")
    idxs = ", ".join(str(idx) for idx, _ in matches)
    _cli_error(f"old_str matched multiple cells: {idxs}. Use --cell_id or a more specific old_str.")

In [ ]:
#| export
def _replace_cell(nb, idx, new_cell):
    old_id = getattr(nb.cells[idx], "id", None)
    if old_id is not None: new_cell.id = old_id
    nb.cells[idx] = new_cell

In [ ]:
#| export
def _clear_outputs(cell):
    if getattr(cell, "cell_type", None) == "code":
        cell.outputs = []
        cell.execution_count = None
    return cell

In [ ]:
#| export
def _load_cells_text(cells="", cells_file=None):
    if cells_file:
        if cells: raise ValueError("Use either cells or cells_file, not both")
        return Path(cells_file).expanduser().read_text(encoding="utf-8")
    if cells == "-": return sys.stdin.read()
    return cells

In [ ]:
#| export
def _should_validate_python(source):
    for line in source.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(("%", "!")): return False
    return bool(source.strip())

In [ ]:
#| export
def _format_syntax_error(source, err, cell_idx):
    lines = source.splitlines()
    line = lines[err.lineno - 1] if err.lineno and 0 < err.lineno <= len(lines) else ""
    pointer = " " * max((err.offset or 1) - 1, 0) + "^" if line else ""
    msg = [f"Invalid Python in new code cell {cell_idx}: {err.msg} at line {err.lineno}, column {err.offset}"]
    if line: msg += [line, pointer]
    msg.append("Tip: shell quoting can turn backslash-n escapes into real newlines inside Python strings. Use --cells_file PATH or cells=- for complex code.")
    return chr(10).join(msg)

In [ ]:
#| export
def _validate_code_cells(cells):
    for idx, cell in enumerate(cells):
        cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
        if cell_type != "code": continue
        source = _cell_source(cell)
        if not _should_validate_python(source): continue
        try: ast.parse(source)
        except SyntaxError as err:
            msg = _format_syntax_error(source, err, idx)
            if _in_call_parse.get(): raise SystemExit(msg)
            raise ValueError(msg) from err

In [ ]:
#| export
def _parse_cells(cells, default_type="code"):
    if isinstance(cells, (list, tuple)): return _split_symbol_cells([_coerce_cell(o, default_type) for o in cells])

    parsed = []
    for block in _split_blocks(cells):
        lines = block.splitlines()
        marker = lines[0].strip().lower() if lines else ""
        cell_type = default_type
        if marker in {"%%markdown", "%%md"}:
            cell_type, lines = "markdown", lines[1:]
        elif marker == "%%code":
            cell_type, lines = "code", lines[1:]
        elif marker == "%%raw":
            cell_type, lines = "raw", lines[1:]
        parsed.append(mk_cell("\n".join(lines), cell_type=cell_type))
    return _split_symbol_cells(parsed)

In [ ]:
#| export
def _first_line(source):
    for line in source.splitlines():
        line = line.strip()
        if line: return line
    return ""

In [ ]:
#| export
def _cell_prefix(idx, cell, show_ids=False):
    suffix = f" hash={_cell_hash(cell)}" if show_ids else ""
    return f"Cell id={cell.id}{suffix}: {cell.cell_type}"

In [ ]:
#| export
def _format_chapter_spans(spans, cells, show_ids=False):
    lines = []
    for span in spans:
        cell = cells[span["start"]]
        suffix = f" hash={_cell_hash(cell)}" if show_ids else ""
        lines.append(f"Chapter id={cell.id}{suffix}: ## {span['title']}")
    return "\n".join(lines)

In [ ]:
#| export
def _matches_filter(source, pattern):
    pattern = str(pattern)
    if pattern in source: return True
    try: return re.search(pattern, source, flags=re.MULTILINE) is not None
    except re.error: return False

In [ ]:
#| export
def _is_exported_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    return any(_is_export_directive(line) for line in _cell_source(cell).splitlines())


def _normalize_cell_type_filter(value):
    if value is None: return None
    aliases = {
        "code": "code",
        "py": "code",
        "python": "code",
        "md": "markdown",
        "markdown": "markdown",
        "doc": "markdown",
        "docs": "markdown",
        "raw": "raw",
        "export": "export",
        "exported": "export",
    }
    normalized = set()
    for item in str(value).split(","):
        key = item.strip().lower()
        if not key: continue
        if key not in aliases: raise ValueError(f"Unknown cell_type {item!r}; use code, md, raw, or export")
        normalized.add(aliases[key])
    return normalized or None


def _cell_matches_type(cell, cell_type):
    wanted = _normalize_cell_type_filter(cell_type)
    if wanted is None: return True
    if "export" in wanted and _is_exported_code_cell(cell): return True
    return getattr(cell, "cell_type", None) in (wanted - {"export"})

In [ ]:
#| export
def _with_context(cells, items, context=0):
    context = int(context or 0)
    if context < 0: raise ValueError("context must be >= 0")
    if context == 0: return items

    idxs = {idx for idx, _ in items}
    for idx in list(idxs):
        for offset in range(1, context + 1):
            prev = idx - offset
            if prev < 0: break
            if cells[prev].cell_type == "markdown": idxs.add(prev)
        for offset in range(1, context + 1):
            nxt = idx + offset
            if nxt >= len(cells): break
            if cells[nxt].cell_type == "code" and not _is_exported_code_cell(cells[nxt]): idxs.add(nxt)
    return [(idx, cells[idx]) for idx in sorted(idxs)]

In [ ]:
#| export
def _chapter_title(cell):
    if getattr(cell, "cell_type", None) != "markdown": return None
    for line in _cell_source(cell).splitlines():
        match = re.match(r"^##\s+(.+?)\s*$", line.strip())
        if match: return match.group(1).strip()
    return None

In [ ]:
#| export
def _chapter_spans(cells):
    starts = [(idx, title) for idx, cell in enumerate(cells) if (title := _chapter_title(cell))]
    spans = []
    for pos, (start, title) in enumerate(starts):
        end = starts[pos + 1][0] if pos + 1 < len(starts) else len(cells)
        spans.append(dict(title=title, start=start, end=end))
    return spans

In [ ]:
#| export
def _matching_chapters(cells, chapter=None):
    spans = _chapter_spans(cells)
    if chapter is None: return spans
    return [span for span in spans if _matches_filter(span["title"], chapter)]

In [ ]:
#| export
def _chapter_index_set(cells, chapter):
    idxs = set()
    for span in _matching_chapters(cells, chapter):
        idxs.update(range(span["start"], span["end"]))
    return idxs

In [ ]:
#| export
def _one_chapter(cells, chapter, create=False):
    matches = _matching_chapters(cells, chapter)
    if len(matches) == 1: return matches[0]
    if not matches and create:
        cells.append(mk_cell(f"## {chapter}", cell_type="markdown"))
        return dict(title=str(chapter), start=len(cells) - 1, end=len(cells))
    if not matches: raise ValueError(f"No chapter matches {chapter!r}")
    titles = ", ".join(f"{span['title']} ({span['start']}:{span['end']})" for span in matches)
    raise ValueError(f"Chapter {chapter!r} matches multiple chapters: {titles}")

In [ ]:
#| export
def _chapter_body_len(span):
    return max(span["end"] - span["start"] - 1, 0)

In [ ]:
#| export
def _chapter_body_slice(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    start, stop, step = target.indices(body_len)
    if step != 1: raise ValueError("chapter ranges do not support steps")
    return slice(body_start + start, body_start + stop)

In [ ]:
#| export
def _chapter_delete(cells, span, selector):
    if selector is None: return
    target = _parse_write_target(selector)
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    if isinstance(target, slice):
        del cells[_chapter_body_slice(span, target)]
        return
    idx = int(target)
    if idx < 0: idx += body_len
    if idx < 0 or idx >= body_len: raise IndexError(target)
    del cells[body_start + idx]

In [ ]:
#| export
def _chapter_insert_target(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    if target is None: return slice(body_start, body_start + body_len)
    if isinstance(target, slice): return _chapter_body_slice(span, target)
    idx = int(target)
    if idx == -1: return body_start + body_len
    if idx < 0: idx += body_len
    if idx < 0 or idx > body_len: raise IndexError(target)
    return body_start + idx

In [0]:
import tempfile as _tempfile
from pathlib import Path as _Path

from nbskill.foundation import _cell_hash, _failure_map_path, _parse_cells

with _tempfile.TemporaryDirectory() as td:
    custom_map = _Path(td) / "errors.json"
    import os as _os
    _os.environ["NBSKILL_FAILURE_MAP"] = str(custom_map)
    assert _failure_map_path() == custom_map

cells = _parse_cells("%%markdown\n# Note\n---\n%%code\nanswer = 42")
assert [cell.cell_type for cell in cells] == ["markdown", "code"]
assert _cell_hash("answer = 42")